# Credit Risk Model Evaluation

This notebook provides a comprehensive analysis of model performance across credit risk datasets for both:
- **PD (Probability of Default)**: Classification task - primary metric: AUC-ROC
- **LGD (Loss Given Default)**: Regression task - primary metric: R²

The evaluation covers multiple models tested across datasets with and without hyperparameter optimization (HPO).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100

# =============================================================================
# CONFIGURATION - Update these paths as needed
# =============================================================================
EXPERIMENT_NAME = 'experiment1'

def find_results_directory(experiment_name: str) -> Path:
    """
    Try to find the results directory in multiple possible locations.
    """
    # Get various reference points
    cwd = Path.cwd()
    
    # Try to get notebook directory (works in Jupyter)
    try:
        notebook_dir = Path(os.path.dirname(os.path.abspath('__file__')))
    except:
        notebook_dir = cwd
    
    # List of possible locations to check
    possible_paths = [
        # Relative to current working directory
        cwd / 'results' / experiment_name,
        cwd / experiment_name,
        # Parent directory (if notebook is in scripts/ or notebooks/)
        cwd.parent / 'results' / experiment_name,
        cwd.parent / experiment_name,
        # Two levels up
        cwd.parent.parent / 'results' / experiment_name,
        # Absolute paths (common locations)
        Path(f'./results/{experiment_name}'),
        Path(f'../results/{experiment_name}'),
        Path(f'../../results/{experiment_name}'),
    ]
    
    # Check each path
    for path in possible_paths:
        try:
            if path.exists():
                return path.resolve()
        except:
            continue
    
    return None

# Find the results directory
RESULTS_DIR = find_results_directory(EXPERIMENT_NAME)

if RESULTS_DIR is None:
    print("ERROR: Could not find results directory!")
    print(f"\nSearched for experiment: {EXPERIMENT_NAME}")
    print(f"Current working directory: {Path.cwd()}")
    print("\nPlease either:")
    print("  1. Run this notebook from the project root directory")
    print("  2. Update EXPERIMENT_NAME to match your experiment folder")
    print("  3. Set RESULTS_DIR manually below")
    print("\n# Example manual override:")
    print("# RESULTS_DIR = Path(r'C:/path/to/your/results/experiment1')")
    RESULTS_DIR = Path('.')  # Fallback to current directory
    SUMMARY_DIR = Path('.')
else:
    SUMMARY_DIR = RESULTS_DIR / 'summary'
    print(f"Found results directory: {RESULTS_DIR}")
    print(f"Summary directory: {SUMMARY_DIR}")
    
    # List contents
    if RESULTS_DIR.exists():
        contents = list(RESULTS_DIR.iterdir())
        print(f"\nContents: {[p.name for p in contents]}")

### Manual Path Override (if needed)

If the automatic path detection didn't work, uncomment and modify the cell below:

In [ ]:
# =============================================================================
# MANUAL PATH OVERRIDE - Uncomment and modify if automatic detection fails
# =============================================================================

# For Windows:
# RESULTS_DIR = Path(r'C:\Users\YourName\Projects\TabPFNCredit\results\experiment1')

# For Mac/Linux:
# RESULTS_DIR = Path('/home/user/TabPFNCredit/results/experiment1')

# Relative path from notebook location:
# RESULTS_DIR = Path('../results/experiment1')

# After setting RESULTS_DIR, update SUMMARY_DIR:
# SUMMARY_DIR = RESULTS_DIR / 'summary'
# print(f"Using manual path: {RESULTS_DIR}")

## 1. Load Summary Data

Load the pre-computed summary CSV files generated by `Summarize_Results.py`.

In [ ]:
def load_summary_files(task: str) -> dict:
    """
    Load all summary files for a given task (pd or lgd).
    
    Returns dict with keys: 'no_hpo_per_dataset', 'no_hpo_overall', 'hpo_per_dataset', 'hpo_overall'
    """
    data = {}
    
    for hpo_mode in ['no_hpo', 'hpo']:
        for file_type in ['per_dataset', 'overall']:
            filename = f"summary_{task}_{hpo_mode}_{file_type}.csv"
            filepath = SUMMARY_DIR / filename
            
            key = f"{hpo_mode}_{file_type}"
            
            if filepath.exists():
                data[key] = pd.read_csv(filepath)
                print(f"  Loaded: {filename} ({len(data[key])} rows)")
            else:
                data[key] = pd.DataFrame()
                print(f"  Missing: {filename}")
    
    return data

def load_raw_results(task: str) -> dict:
    """
    Load raw pickle results for detailed fold-level analysis.
    """
    task_dir = RESULTS_DIR / task
    results = {}
    
    if not task_dir.exists():
        print(f"  Task directory not found: {task_dir}")
        return results
    
    for pkl_file in task_dir.glob("*.pkl"):
        dataset_name = pkl_file.stem
        try:
            with open(pkl_file, 'rb') as f:
                results[dataset_name] = pickle.load(f)
        except Exception as e:
            print(f"  Warning: Could not load {dataset_name}: {e}")
    
    print(f"  Loaded {len(results)} dataset result files")
    return results

# Check if we have a valid results directory
if not RESULTS_DIR.exists():
    print(f"ERROR: Results directory does not exist: {RESULTS_DIR}")
    print("Please run Summarize_Results.py first or check the path.")
    pd_summary = {'no_hpo_per_dataset': pd.DataFrame(), 'no_hpo_overall': pd.DataFrame(),
                  'hpo_per_dataset': pd.DataFrame(), 'hpo_overall': pd.DataFrame()}
    lgd_summary = pd_summary.copy()
    pd_raw = {}
    lgd_raw = {}
else:
    print("Loading PD (Classification) data:")
    pd_summary = load_summary_files('pd')
    pd_raw = load_raw_results('pd')
    
    print("\nLoading LGD (Regression) data:")
    lgd_summary = load_summary_files('lgd')
    lgd_raw = load_raw_results('lgd')
    
    # Check if summary files exist, if not suggest running Summarize_Results.py
    all_missing = all(df.empty for df in pd_summary.values()) and all(df.empty for df in lgd_summary.values())
    if all_missing:
        print("\n" + "="*60)
        print("WARNING: No summary files found!")
        print("="*60)
        print("\nPlease run Summarize_Results.py first to generate summary files:")
        print("  python scripts/Summarize_Results.py --experiment", EXPERIMENT_NAME)
        print("\nOr check that your experiment has completed successfully.")

## 2. Data Overview

Overview of available data for both tasks.

In [ ]:
def print_data_overview(task_name: str, summary_data: dict, raw_data: dict, primary_metric: str):
    """
    Print overview of available data for a task.
    """
    print(f"{'='*60}")
    print(f" {task_name} Data Overview")
    print(f"{'='*60}")
    
    # Check overall summary
    for hpo_mode in ['no_hpo', 'hpo']:
        overall_key = f"{hpo_mode}_overall"
        per_dataset_key = f"{hpo_mode}_per_dataset"
        
        if not summary_data[overall_key].empty:
            df = summary_data[overall_key]
            print(f"\n{hpo_mode.upper().replace('_', ' ')}:")
            print(f"  Methods: {len(df)}")
            print(f"  Methods available: {', '.join(df['Method'].tolist())}")
            
            if not summary_data[per_dataset_key].empty:
                datasets = summary_data[per_dataset_key]['Dataset'].unique()
                print(f"  Datasets: {len(datasets)}")
    
    print(f"\nRaw result files: {len(raw_data)}")
    if raw_data:
        print(f"  Datasets: {', '.join(list(raw_data.keys())[:5])}..." if len(raw_data) > 5 else f"  Datasets: {', '.join(raw_data.keys())}")

print_data_overview("PD (Classification)", pd_summary, pd_raw, 'AUC')
print()
print_data_overview("LGD (Regression)", lgd_summary, lgd_raw, 'R2')

---
# Part A: PD (Probability of Default) Analysis
---

Classification task evaluation using AUC-ROC as the primary metric.

## A.1 Overall Model Performance (PD)

In [ ]:
def plot_overall_performance(summary_data: dict, primary_metric: str, task_name: str):
    """
    Plot overall model performance comparison between NO_HPO and HPO.
    """
    mean_col = f"{primary_metric}_mean"
    std_col = f"{primary_metric}_std"
    
    no_hpo_df = summary_data['no_hpo_overall']
    hpo_df = summary_data['hpo_overall']
    
    if no_hpo_df.empty and hpo_df.empty:
        print(f"No data available for {task_name}")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for idx, (df, title, ax) in enumerate([
        (no_hpo_df, 'Without HPO', axes[0]),
        (hpo_df, 'With HPO (Optuna)', axes[1])
    ]):
        if df.empty or mean_col not in df.columns:
            ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{task_name} - {title}')
            continue
        
        df_sorted = df.sort_values(mean_col, ascending=False)
        methods = df_sorted['Method'].tolist()
        means = df_sorted[mean_col].values
        stds = df_sorted[std_col].values if std_col in df_sorted.columns else np.zeros_like(means)
        
        bars = ax.bar(methods, means, yerr=stds, capsize=3, alpha=0.8, color='steelblue', edgecolor='black')
        
        # Adjust y-axis
        valid_means = means[~np.isnan(means)]
        if len(valid_means) > 0:
            padding = (valid_means.max() - valid_means.min()) * 0.1
            ax.set_ylim(max(0, valid_means.min() - padding), valid_means.max() + padding)
        
        ax.set_ylabel(primary_metric)
        ax.set_title(f'{task_name} - {title}', fontsize=14, fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        ax.yaxis.grid(True, alpha=0.3)  # Only horizontal grid lines
        
        # Add value labels
        for bar, mean in zip(bars, means):
            if not np.isnan(mean):
                ax.annotate(f'{mean:.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                        ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.show()

# Plot PD performance
plot_overall_performance(pd_summary, 'AUC', 'PD (Classification)')

## A.2 HPO Impact Analysis (PD)

In [ ]:
def plot_hpo_comparison(summary_data: dict, primary_metric: str, task_name: str):
    """
    Compare NO_HPO vs HPO performance (like the reference screenshot).
    """
    mean_col = f"{primary_metric}_mean"
    
    no_hpo_df = summary_data['no_hpo_overall']
    hpo_df = summary_data['hpo_overall']
    
    if no_hpo_df.empty or hpo_df.empty:
        print(f"Cannot compare HPO: missing data for {task_name}")
        return None
    
    if mean_col not in no_hpo_df.columns or mean_col not in hpo_df.columns:
        print(f"Primary metric {primary_metric} not found")
        return None
    
    # Merge dataframes
    merged = no_hpo_df[['Method', mean_col]].merge(
        hpo_df[['Method', mean_col]],
        on='Method',
        suffixes=('_NO_HPO', '_HPO')
    )
    
    if merged.empty:
        print("No matching methods between HPO and NO_HPO")
        return None
    
    # Sort by HPO performance
    merged = merged.sort_values(f'{mean_col}_HPO', ascending=False)
    
    # Create overlapping bar chart (like screenshot)
    fig, ax = plt.subplots(figsize=(14, 7))
    
    methods = merged['Method'].tolist()
    no_hpo_vals = merged[f'{mean_col}_NO_HPO'].values
    hpo_vals = merged[f'{mean_col}_HPO'].values
    
    x = np.arange(len(methods))
    width = 0.7
    
    # "No Tuning" bars (with red edge)
    bars_no_hpo = ax.bar(x, no_hpo_vals, width, label='No Tuning',
                         color='#AEC7E8', edgecolor='#E74C3C', linewidth=2, alpha=0.6)
    
    # "Optuna" bars on top
    bars_hpo = ax.bar(x, hpo_vals, width * 0.85, label='Optuna',
                      color='#1F77B4', alpha=0.85)
    
    ax.set_ylabel(f'Average {primary_metric.lower()}', fontsize=12)
    ax.set_xlabel('Learning Algorithms', fontsize=12)
    ax.set_title(f'{task_name} - Average {primary_metric} Before and After Optuna HPO', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=45, ha='right')
    ax.set_ylim(0, max(max(no_hpo_vals), max(hpo_vals)) * 1.05)
    ax.legend(loc='upper right', fontsize=10)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return merged

# Plot HPO comparison for PD
pd_hpo_comparison = plot_hpo_comparison(pd_summary, 'AUC', 'PD (Classification)')

In [ ]:
def plot_hpo_improvement(comparison_df: pd.DataFrame, primary_metric: str, task_name: str):
    """
    Plot improvement from HPO for each method.
    """
    if comparison_df is None or comparison_df.empty:
        print("No comparison data available")
        return
    
    mean_col = f"{primary_metric}_mean"
    
    # Calculate improvement
    comparison_df = comparison_df.copy()
    comparison_df['Improvement'] = comparison_df[f'{mean_col}_HPO'] - comparison_df[f'{mean_col}_NO_HPO']
    comparison_df['Pct_Improvement'] = (comparison_df['Improvement'] / comparison_df[f'{mean_col}_NO_HPO']) * 100
    
    # Sort by improvement
    comparison_df = comparison_df.sort_values('Improvement', ascending=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Absolute improvement
    colors = ['#27AE60' if imp >= 0 else '#C0392B' for imp in comparison_df['Improvement']]
    axes[0].barh(comparison_df['Method'], comparison_df['Improvement'], color=colors, alpha=0.8)
    axes[0].axvline(x=0, color='black', linestyle='-', linewidth=1)
    axes[0].set_xlabel(f'{primary_metric} Improvement (HPO - No HPO)')
    axes[0].set_title(f'{task_name} - Absolute Improvement from HPO', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Percentage improvement
    colors_pct = ['#27AE60' if imp >= 0 else '#C0392B' for imp in comparison_df['Pct_Improvement']]
    axes[1].barh(comparison_df['Method'], comparison_df['Pct_Improvement'], color=colors_pct, alpha=0.8)
    axes[1].axvline(x=0, color='black', linestyle='-', linewidth=1)
    axes[1].set_xlabel('Percentage Improvement (%)')
    axes[1].set_title(f'{task_name} - Percentage Improvement from HPO', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print(f"\n=== HPO IMPROVEMENT SUMMARY ({task_name}) ===")
    improved = (comparison_df['Improvement'] > 0).sum()
    total = len(comparison_df)
    print(f"Methods improved by HPO: {improved}/{total} ({improved/total*100:.1f}%)")
    print(f"Average improvement: {comparison_df['Improvement'].mean():.4f} ({comparison_df['Pct_Improvement'].mean():.2f}%)")
    
    print(f"\nTop 3 most improved:")
    for _, row in comparison_df.nlargest(3, 'Improvement').iterrows():
        print(f"  {row['Method']}: +{row['Improvement']:.4f} (+{row['Pct_Improvement']:.2f}%)")
    
    print(f"\nTop 3 most declined:")
    for _, row in comparison_df.nsmallest(3, 'Improvement').iterrows():
        print(f"  {row['Method']}: {row['Improvement']:.4f} ({row['Pct_Improvement']:.2f}%)")

# Plot HPO improvement for PD
plot_hpo_improvement(pd_hpo_comparison, 'AUC', 'PD (Classification)')

## A.3 Dataset-Level Performance (PD)

In [ ]:
def plot_dataset_heatmap(summary_data: dict, primary_metric: str, task_name: str, hpo_mode: str = 'no_hpo'):
    """
    Plot heatmap of method performance across datasets.
    """
    mean_col = f"{primary_metric}_mean"
    per_dataset_df = summary_data[f'{hpo_mode}_per_dataset']
    
    if per_dataset_df.empty or mean_col not in per_dataset_df.columns:
        print(f"No per-dataset data available for {task_name} ({hpo_mode})")
        return
    
    # Pivot: methods as rows, datasets as columns
    pivot = per_dataset_df.pivot(index='Method', columns='Dataset', values=mean_col)
    
    # Sort by mean performance
    pivot['_mean'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('_mean', ascending=False)
    pivot = pivot.drop('_mean', axis=1)
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(max(14, len(pivot.columns) * 0.8), max(8, len(pivot) * 0.5)))
    
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
                cbar_kws={'label': primary_metric})
    
    hpo_label = 'Without HPO' if hpo_mode == 'no_hpo' else 'With HPO'
    ax.set_title(f'{task_name} - {primary_metric} Across Datasets ({hpo_label})', fontsize=14, fontweight='bold')
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Method')
    
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    # Print best model per dataset
    print(f"\nBest Method per Dataset ({hpo_label}):")
    for dataset in pivot.columns:
        best_method = pivot[dataset].idxmax()
        best_score = pivot[dataset].max()
        print(f"  {dataset}: {best_method} ({primary_metric}: {best_score:.4f})")

# Plot PD dataset heatmap
plot_dataset_heatmap(pd_summary, 'AUC', 'PD (Classification)', 'no_hpo')

## A.4 Model Ranking Analysis (PD)

In [ ]:
def compute_and_plot_rankings(summary_data: dict, primary_metric: str, task_name: str):
    """
    Compute average rank across datasets and plot.
    """
    mean_col = f"{primary_metric}_mean"
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for idx, hpo_mode in enumerate(['no_hpo', 'hpo']):
        per_dataset_df = summary_data[f'{hpo_mode}_per_dataset']
        ax = axes[idx]
        
        if per_dataset_df.empty or mean_col not in per_dataset_df.columns:
            ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes)
            continue
        
        # Calculate rank within each dataset
        def rank_within_dataset(group):
            group = group.copy()
            group['Rank'] = group[mean_col].rank(ascending=False, method='min')
            return group
        
        ranked_df = per_dataset_df.groupby('Dataset', group_keys=False).apply(rank_within_dataset)
        avg_rank = ranked_df.groupby('Method')['Rank'].mean().sort_values()
        
        # Plot
        colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(avg_rank)))
        ax.barh(avg_rank.index, avg_rank.values, color=colors, alpha=0.8)
        ax.axvline(x=avg_rank.mean(), color='red', linestyle='--', alpha=0.7, label=f'Mean: {avg_rank.mean():.2f}')
        
        hpo_label = 'Without HPO' if hpo_mode == 'no_hpo' else 'With HPO'
        ax.set_xlabel('Average Rank (lower is better)')
        ax.set_title(f'{task_name} - {hpo_label}', fontsize=12, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{task_name} - Average Rank Across Datasets', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Plot PD rankings
compute_and_plot_rankings(pd_summary, 'AUC', 'PD (Classification)')

## A.5 Multi-Metric Comparison (PD)

In [ ]:
def plot_metrics_heatmap(summary_data: dict, metrics: list, task_name: str, hpo_mode: str = 'no_hpo'):
    """
    Plot heatmap of multiple metrics for each method.
    """
    overall_df = summary_data[f'{hpo_mode}_overall']
    
    if overall_df.empty:
        print(f"No overall data available for {task_name} ({hpo_mode})")
        return
    
    # Get mean columns
    mean_cols = [f'{m}_mean' for m in metrics if f'{m}_mean' in overall_df.columns]
    
    if not mean_cols:
        print(f"No metric columns found")
        return
    
    # Create pivot table
    pivot_data = overall_df.set_index('Method')[mean_cols].copy()
    pivot_data.columns = [c.replace('_mean', '') for c in pivot_data.columns]
    
    # Sort by first metric
    pivot_data = pivot_data.sort_values(pivot_data.columns[0], ascending=False)
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(max(10, len(mean_cols) * 1.5), max(8, len(pivot_data) * 0.4)))
    
    sns.heatmap(pivot_data, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
                cbar_kws={'label': 'Score'})
    
    hpo_label = 'Without HPO' if hpo_mode == 'no_hpo' else 'With HPO'
    ax.set_title(f'{task_name} - Method × Metric ({hpo_label})', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# PD metrics
pd_metrics = ['AUC', 'Accuracy', 'F1', 'LogLoss', 'Avg_Precision', 'Avg_Recall']
plot_metrics_heatmap(pd_summary, pd_metrics, 'PD (Classification)', 'no_hpo')

---
# Part B: LGD (Loss Given Default) Analysis
---

Regression task evaluation using R² as the primary metric.

## B.1 Overall Model Performance (LGD)

In [ ]:
# Plot LGD performance
plot_overall_performance(lgd_summary, 'R2', 'LGD (Regression)')

## B.2 HPO Impact Analysis (LGD)

In [ ]:
# Plot HPO comparison for LGD
lgd_hpo_comparison = plot_hpo_comparison(lgd_summary, 'R2', 'LGD (Regression)')

In [ ]:
# Plot HPO improvement for LGD
plot_hpo_improvement(lgd_hpo_comparison, 'R2', 'LGD (Regression)')

## B.3 Dataset-Level Performance (LGD)

In [ ]:
# Plot LGD dataset heatmap
plot_dataset_heatmap(lgd_summary, 'R2', 'LGD (Regression)', 'no_hpo')

## B.4 Model Ranking Analysis (LGD)

In [ ]:
# Plot LGD rankings
compute_and_plot_rankings(lgd_summary, 'R2', 'LGD (Regression)')

## B.5 Multi-Metric Comparison (LGD)

In [ ]:
# LGD metrics
lgd_metrics = ['R2', 'RMSE', 'MAE', 'MSE']
plot_metrics_heatmap(lgd_summary, lgd_metrics, 'LGD (Regression)', 'no_hpo')

---
# Part C: Cross-Task Comparison
---

In [ ]:
def compare_tasks_summary(pd_summary: dict, lgd_summary: dict):
    """
    Compare top performers across both tasks.
    """
    print("=" * 80)
    print(" CROSS-TASK COMPARISON SUMMARY")
    print("=" * 80)
    
    for hpo_mode in ['no_hpo', 'hpo']:
        hpo_label = 'Without HPO' if hpo_mode == 'no_hpo' else 'With HPO'
        print(f"\n{'='*40}")
        print(f" {hpo_label}")
        print(f"{'='*40}")
        
        # PD Top 5
        pd_df = pd_summary[f'{hpo_mode}_overall']
        if not pd_df.empty and 'AUC_mean' in pd_df.columns:
            print(f"\nPD (Classification) - Top 5 by AUC:")
            top_pd = pd_df.nlargest(5, 'AUC_mean')
            for i, (_, row) in enumerate(top_pd.iterrows(), 1):
                print(f"  {i}. {row['Method']}: {row['AUC_mean']:.4f}")
        
        # LGD Top 5
        lgd_df = lgd_summary[f'{hpo_mode}_overall']
        if not lgd_df.empty and 'R2_mean' in lgd_df.columns:
            print(f"\nLGD (Regression) - Top 5 by R²:")
            top_lgd = lgd_df.nlargest(5, 'R2_mean')
            for i, (_, row) in enumerate(top_lgd.iterrows(), 1):
                print(f"  {i}. {row['Method']}: {row['R2_mean']:.4f}")

compare_tasks_summary(pd_summary, lgd_summary)

In [ ]:
def plot_side_by_side_comparison(pd_summary: dict, lgd_summary: dict, hpo_mode: str = 'no_hpo'):
    """
    Side-by-side comparison of top methods for both tasks.
    """
    pd_df = pd_summary[f'{hpo_mode}_overall']
    lgd_df = lgd_summary[f'{hpo_mode}_overall']
    
    if pd_df.empty and lgd_df.empty:
        print("No data available for comparison")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    hpo_label = 'Without HPO' if hpo_mode == 'no_hpo' else 'With HPO'
    
    # PD
    if not pd_df.empty and 'AUC_mean' in pd_df.columns:
        pd_sorted = pd_df.sort_values('AUC_mean', ascending=True)
        axes[0].barh(pd_sorted['Method'], pd_sorted['AUC_mean'], color='steelblue', alpha=0.8)
        axes[0].set_xlabel('AUC')
        axes[0].set_title(f'PD (Classification) - AUC\n{hpo_label}', fontsize=12, fontweight='bold')
        axes[0].grid(True, alpha=0.3)
    else:
        axes[0].text(0.5, 0.5, 'No PD data', ha='center', va='center', transform=axes[0].transAxes)
    
    # LGD
    if not lgd_df.empty and 'R2_mean' in lgd_df.columns:
        lgd_sorted = lgd_df.sort_values('R2_mean', ascending=True)
        axes[1].barh(lgd_sorted['Method'], lgd_sorted['R2_mean'], color='coral', alpha=0.8)
        axes[1].set_xlabel('R²')
        axes[1].set_title(f'LGD (Regression) - R²\n{hpo_label}', fontsize=12, fontweight='bold')
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, 'No LGD data', ha='center', va='center', transform=axes[1].transAxes)
    
    plt.suptitle(f'Cross-Task Performance Comparison ({hpo_label})', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Plot side-by-side comparison
plot_side_by_side_comparison(pd_summary, lgd_summary, 'no_hpo')
plot_side_by_side_comparison(pd_summary, lgd_summary, 'hpo')

---
# Part D: Statistical Analysis
---

In [ ]:
def statistical_hpo_test(summary_data: dict, primary_metric: str, task_name: str):
    """
    Perform statistical tests comparing HPO vs NO_HPO performance.
    """
    mean_col = f"{primary_metric}_mean"
    
    no_hpo_df = summary_data['no_hpo_overall']
    hpo_df = summary_data['hpo_overall']
    
    if no_hpo_df.empty or hpo_df.empty:
        print(f"Cannot perform statistical test: missing data for {task_name}")
        return
    
    if mean_col not in no_hpo_df.columns or mean_col not in hpo_df.columns:
        print(f"Primary metric {primary_metric} not found")
        return
    
    # Merge to get paired data
    merged = no_hpo_df[['Method', mean_col]].merge(
        hpo_df[['Method', mean_col]],
        on='Method',
        suffixes=('_NO_HPO', '_HPO')
    )
    
    if len(merged) < 2:
        print("Not enough paired data for statistical test")
        return
    
    no_hpo_scores = merged[f'{mean_col}_NO_HPO'].values
    hpo_scores = merged[f'{mean_col}_HPO'].values
    
    # Paired t-test
    t_stat, p_value = stats.ttest_rel(hpo_scores, no_hpo_scores)
    
    # Wilcoxon signed-rank test (non-parametric)
    try:
        w_stat, w_p_value = stats.wilcoxon(hpo_scores, no_hpo_scores)
    except:
        w_stat, w_p_value = np.nan, np.nan
    
    print(f"\n{'='*60}")
    print(f" STATISTICAL ANALYSIS: {task_name}")
    print(f"{'='*60}")
    print(f"\nMetric: {primary_metric}")
    print(f"Number of methods compared: {len(merged)}")
    print(f"\nDescriptive Statistics:")
    print(f"  NO HPO mean: {no_hpo_scores.mean():.4f} (±{no_hpo_scores.std():.4f})")
    print(f"  HPO mean:    {hpo_scores.mean():.4f} (±{hpo_scores.std():.4f})")
    print(f"  Difference:  {(hpo_scores.mean() - no_hpo_scores.mean()):.4f}")
    
    print(f"\nPaired t-test:")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
    print(f"  Significance: {sig}")
    
    if not np.isnan(w_p_value):
        print(f"\nWilcoxon signed-rank test:")
        print(f"  W-statistic: {w_stat:.4f}")
        print(f"  p-value: {w_p_value:.6f}")
        w_sig = '***' if w_p_value < 0.001 else '**' if w_p_value < 0.01 else '*' if w_p_value < 0.05 else 'ns'
        print(f"  Significance: {w_sig}")

# Statistical tests
statistical_hpo_test(pd_summary, 'AUC', 'PD (Classification)')
statistical_hpo_test(lgd_summary, 'R2', 'LGD (Regression)')

---
# Part E: Summary Tables
---

In [ ]:
def display_summary_tables(pd_summary: dict, lgd_summary: dict):
    """
    Display formatted summary tables.
    """
    print("\n" + "=" * 80)
    print(" SUMMARY TABLES")
    print("=" * 80)
    
    for task_name, summary_data, primary_metric in [
        ('PD (Classification)', pd_summary, 'AUC'),
        ('LGD (Regression)', lgd_summary, 'R2')
    ]:
        print(f"\n{'='*60}")
        print(f" {task_name}")
        print(f"{'='*60}")
        
        for hpo_mode in ['no_hpo', 'hpo']:
            df = summary_data[f'{hpo_mode}_overall']
            hpo_label = 'Without HPO' if hpo_mode == 'no_hpo' else 'With HPO'
            
            if df.empty:
                print(f"\n{hpo_label}: No data available")
                continue
            
            mean_col = f'{primary_metric}_mean'
            std_col = f'{primary_metric}_std'
            
            if mean_col in df.columns:
                df_sorted = df.sort_values(mean_col, ascending=False).copy()
                
                print(f"\n{hpo_label} - Sorted by {primary_metric}:")
                
                # Create display dataframe
                display_cols = ['Method']
                for metric in [primary_metric, 'Accuracy', 'F1', 'RMSE', 'MAE']:
                    mc = f'{metric}_mean'
                    sc = f'{metric}_std'
                    if mc in df_sorted.columns:
                        display_cols.extend([mc, sc])
                
                available_cols = [c for c in display_cols if c in df_sorted.columns]
                print(df_sorted[available_cols].to_string(index=False))

display_summary_tables(pd_summary, lgd_summary)

---
# Final Summary
---

In [ ]:
def final_summary(pd_summary: dict, lgd_summary: dict):
    """
    Print final evaluation summary.
    """
    print("=" * 80)
    print(" FINAL EVALUATION SUMMARY")
    print("=" * 80)
    
    # PD Summary
    print("\n" + "-" * 40)
    print(" PD (Probability of Default)")
    print("-" * 40)
    
    pd_no_hpo = pd_summary['no_hpo_overall']
    pd_hpo = pd_summary['hpo_overall']
    
    if not pd_no_hpo.empty and 'AUC_mean' in pd_no_hpo.columns:
        best_no_hpo = pd_no_hpo.loc[pd_no_hpo['AUC_mean'].idxmax()]
        print(f"\nBest method (NO HPO): {best_no_hpo['Method']} (AUC: {best_no_hpo['AUC_mean']:.4f})")
    
    if not pd_hpo.empty and 'AUC_mean' in pd_hpo.columns:
        best_hpo = pd_hpo.loc[pd_hpo['AUC_mean'].idxmax()]
        print(f"Best method (HPO):    {best_hpo['Method']} (AUC: {best_hpo['AUC_mean']:.4f})")
    
    # LGD Summary
    print("\n" + "-" * 40)
    print(" LGD (Loss Given Default)")
    print("-" * 40)
    
    lgd_no_hpo = lgd_summary['no_hpo_overall']
    lgd_hpo = lgd_summary['hpo_overall']
    
    if not lgd_no_hpo.empty and 'R2_mean' in lgd_no_hpo.columns:
        best_no_hpo = lgd_no_hpo.loc[lgd_no_hpo['R2_mean'].idxmax()]
        print(f"\nBest method (NO HPO): {best_no_hpo['Method']} (R²: {best_no_hpo['R2_mean']:.4f})")
    
    if not lgd_hpo.empty and 'R2_mean' in lgd_hpo.columns:
        best_hpo = lgd_hpo.loc[lgd_hpo['R2_mean'].idxmax()]
        print(f"Best method (HPO):    {best_hpo['Method']} (R²: {best_hpo['R2_mean']:.4f})")
    
    print("\n" + "=" * 80)
    print(f" Experiment: {EXPERIMENT_NAME}")
    print(f" Results directory: {RESULTS_DIR}")
    print("=" * 80)

final_summary(pd_summary, lgd_summary)